# IOAI — 2026 Contest Image Classification (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-contest-image-classification/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Image Classification — 모범답안 (동결 ResNet18 + 선형 헤드)

ImageNet 사전학습 **ResNet18** 백본을 동결(frozen)해 특징을 뽑고 그 위에 12-클래스 선형 헤드만 학습한다
(이미지가 적어 과적합 방지). accuracy ≈ 0.75 (최빈 0.08 대비). 더 끌어올리려면 백본 부분 파인튜닝.

In [ ]:
import pandas as pd, os
train = pd.read_csv("data/train.csv")            # id, label
test_ids = pd.read_csv("data/test_list.csv")["id"].tolist()
labels = sorted(train.label.unique())
print("train", len(train), "| classes", len(labels), "| test", len(test_ids))

In [ ]:
import torch, torch.nn as nn
from PIL import Image
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, TensorDataset
dev="cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0)
l2i={l:i for i,l in enumerate(labels)}
norm=transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
tf=transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), norm])
def load_imgs(ids, folder):
    return torch.stack([tf(Image.open(f"data/{folder}/{i}.jpg").convert("RGB")) for i in ids])
Xtr=load_imgs(train.id.tolist(),"train"); ytr=torch.tensor([l2i[l] for l in train.label])
Xte=load_imgs(test_ids,"test")

net=models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in net.parameters(): p.requires_grad=False        # 백본 동결
net.fc=nn.Linear(net.fc.in_features, len(labels)); net=net.to(dev)
dl=DataLoader(TensorDataset(Xtr,ytr), batch_size=64, shuffle=True)
opt=torch.optim.AdamW(net.fc.parameters(), 1e-3); lossf=nn.CrossEntropyLoss()
for ep in range(5):
    net.train()
    for x,y in dl: opt.zero_grad(); lossf(net(x.to(dev)),y.to(dev)).backward(); opt.step()
    print("epoch",ep,"done")
net.eval()
with torch.no_grad(): pr=net(Xte.to(dev)).argmax(1).cpu().numpy()
pred=[labels[i] for i in pr]
pd.DataFrame({"id": test_ids, "label": pred}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(pred))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)